# Feature Discovery Example

This notebook demonstrates the proper workflow for feature discovery in Galactus.

## Core Principles

1. **Features are experimental** - They must not contaminate production
2. **Hypothesis-driven** - Every feature needs a clear hypothesis
3. **Isolated** - Features must be self-contained
4. **Documented** - Assumptions and failure modes must be explicit

## Prerequisites

- Read: `docs/06-research-framework/feature-discovery-rules.md`
- Read: `docs/06-research-framework/research-methodology.md`
- Understand: Features stay in Python until promoted


## Setup

In [1]:
# Ensure we're NOT in production environment
import os
import sys

# Add src to path for imports
sys.path.insert(0, '../../src')

# Verify environment
if os.getenv('GALACTUS_ENV') == 'production':
    raise RuntimeError("Cannot run feature discovery in production!")

print("✓ Environment check passed")
print("✓ Running in research mode")

✓ Environment check passed
✓ Running in research mode


## Import Feature Discovery Tools

In [2]:
# Import isolation utilities
from features.isolation import (
    mark_experimental,
    FeatureIsolationContext,
    list_experimental_features,
    get_feature_metadata
)

# Import example features
from features.examples import (
    compute_oi_decay_pressure,
    compute_hedge_pressure,
    compute_basis_pressure,
    get_feature_catalog
)

print("✓ Imports successful")

✓ Imports successful


## List Available Features

Before using features, let's see what's available:

In [3]:
import features.examples as examples_module

# List all experimental features
all_features = list_experimental_features(examples_module)

print(f"Found {len(all_features)} experimental features:\n")

for name, metadata in all_features.items():
    print(f"📦 {name}")
    print(f"   Hypothesis: {metadata.hypothesis}")
    print(f"   Assumptions: {len(metadata.assumptions)} documented")
    print(f"   Failure Modes: {len(metadata.failure_modes)} documented")
    print(f"   Promoted: {'Yes' if metadata.promoted else 'No'}")
    print()

Found 3 experimental features:

📦 compute_basis_pressure
   Hypothesis: Futures basis divergence indicates forced arbitrage activity
   Assumptions: 4 documented
   Failure Modes: 4 documented
   Promoted: No

📦 compute_hedge_pressure
   Hypothesis: Call-put OI imbalance at key strikes indicates forced delta hedging
   Assumptions: 4 documented
   Failure Modes: 4 documented
   Promoted: No

📦 compute_oi_decay_pressure
   Hypothesis: Derivatives OI decay rate indicates forced position unwinding
   Assumptions: 4 documented
   Failure Modes: 4 documented
   Promoted: No



## Example 1: Using an Existing Feature

Let's use one of the example features with synthetic data.

In [4]:
# Create synthetic data for testing
previous_oi = {
    18000: 1500,
    18500: 2500,
    19000: 2000,
    19500: 1000
}

current_oi = {
    18000: 1000,  # Decay
    18500: 2000,  # Decay
    19000: 1500,  # Decay
    19500: 800    # Decay
}

# Use the feature within isolation context
with FeatureIsolationContext() as ctx:
    pressure = compute_oi_decay_pressure(
        current_oi=current_oi,
        previous_oi=previous_oi,
        time_delta_hours=1.0
    )
    
    print(f"OI Decay Pressure: {pressure:.4f}")
    print(f"Interpretation: {'Unwinding' if pressure < -0.1 else 'Building' if pressure > 0.1 else 'Neutral'}")

OI Decay Pressure: -1.0000
Interpretation: Unwinding


/tmp/ipykernel_202320/2761065739.py:18: UserWarning: Using experimental feature 'compute_oi_decay_pressure'. Hypothesis: Derivatives OI decay rate indicates forced position unwinding. This feature has NOT been validated for production use.
  pressure = compute_oi_decay_pressure(


## Example 2: Creating a New Feature

In [5]:
@mark_experimental(
    hypothesis="Volume spike with price stagnation indicates absorption by strong hands",
    assumptions=[
        "Tick-level volume data available",
        "Price data is clean and aligned",
        "Normal trading hours"
    ],
    failure_modes=[
        "False signals during news events",
        "May miss gradual accumulation"
    ]
)
def compute_absorption_signal(prices, volumes, window_size=20):
    """Detect absorption patterns from volume-price divergence."""
    if len(prices) < window_size:
        return 0.0
    
    import statistics
    recent_prices = prices[-window_size:]
    recent_volumes = volumes[-window_size:]
    
    price_std = statistics.stdev(recent_prices)
    price_mean = statistics.mean(recent_prices)
    price_cv = price_std / price_mean if price_mean != 0 else 0
    
    avg_volume = statistics.mean(recent_volumes[:-1]) if len(recent_volumes) > 1 else 0
    current_volume = recent_volumes[-1]
    volume_ratio = current_volume / avg_volume if avg_volume > 0 else 0
    
    if price_cv < 0.001 and volume_ratio > 2.0:
        return min(1.0, (volume_ratio - 2.0) / 3.0)
    
    return 0.0

# Test the new feature
test_prices = [18500 + i * 0.5 for i in range(30)]
test_volumes = [1000] * 29 + [5000]

with FeatureIsolationContext() as ctx:
    absorption = compute_absorption_signal(test_prices, test_volumes)
    print(f"Absorption Signal: {absorption:.4f}")

Absorption Signal: 1.0000


/tmp/ipykernel_202320/3331477431.py:40: UserWarning: Using experimental feature 'compute_absorption_signal'. Hypothesis: Volume spike with price stagnation indicates absorption by strong hands. This feature has NOT been validated for production use.
  absorption = compute_absorption_signal(test_prices, test_volumes)


## Example 3: Multi-Feature Analysis

In [6]:
# Synthetic data
calls_oi = {18000: 500, 18500: 1000, 19000: 800}
puts_oi = {18000: 1200, 18500: 1500, 19000: 600}
spot = 18500
futures_price = 18550
days_to_expiry = 15

with FeatureIsolationContext() as ctx:
    hedge_result = compute_hedge_pressure(calls_oi, puts_oi, spot)
    basis_result = compute_basis_pressure(futures_price, spot, days_to_expiry)
    
    print("Multi-Feature Analysis:")
    print(f"\nHedge Pressure: {hedge_result['pressure']:.4f}")
    print(f"Confidence: {hedge_result['confidence']:.4f}")
    print(f"\nBasis Pressure: {basis_result['pressure']:.6f}")
    print(f"Divergence: {basis_result['divergence_pct']:.2f}%")

Multi-Feature Analysis:

Hedge Pressure: -0.1786
Confidence: 0.5600

Basis Pressure: 0.002703
Divergence: 0.27%


/tmp/ipykernel_202320/1338057603.py:9: UserWarning: Using experimental feature 'compute_hedge_pressure'. Hypothesis: Call-put OI imbalance at key strikes indicates forced delta hedging. This feature has NOT been validated for production use.
  hedge_result = compute_hedge_pressure(calls_oi, puts_oi, spot)
/tmp/ipykernel_202320/1338057603.py:10: UserWarning: Using experimental feature 'compute_basis_pressure'. Hypothesis: Futures basis divergence indicates forced arbitrage activity. This feature has NOT been validated for production use.
  basis_result = compute_basis_pressure(futures_price, spot, days_to_expiry)


## Summary

✅ Features are isolated from production  
✅ Every feature has a hypothesis  
✅ Assumptions and failures are documented  
✅ Features emit warnings when used  

**Remember**: Python discovers truth. Rust enforces truth.